# Qdrant Vector Search

This notebook demonstrates vector search on Qdrant using a client class with Bearer token authentication for Domino deployments.

## Configuration

| Variable | Description | Required |
|----------|-------------|----------|
| `QDRANT_URL` | Qdrant server URL | Yes |
| `DOMINO_API_PROXY` | Domino API proxy for auth token | No (for Domino) |

## 1. Setup

In [ ]:
import os
import requests

# Configuration
QDRANT_URL = os.environ.get('QDRANT_URL', 'https://apps.cloud-dogfood.domino.tech/apps/airline-quadrant')
DOMINO_API_PROXY = os.environ.get('DOMINO_API_PROXY', '')

print(f"QDRANT_URL: {QDRANT_URL}")
print(f"DOMINO_API_PROXY: {DOMINO_API_PROXY or 'Not set'}")

## 2. Qdrant Client with Bearer Auth

In [ ]:
class QdrantClient:
    """
    Qdrant client with Bearer token authentication for Domino deployments.
    
    Fetches a fresh access token from DOMINO_API_PROXY for each request
    (tokens are only valid for 5 minutes).
    """
    
    def __init__(self, url: str, domino_api_proxy: str = None):
        self.url = url.rstrip("/")
        self.domino_api_proxy = domino_api_proxy
    
    def _get_token(self) -> str | None:
        """Fetch fresh access token from Domino API proxy."""
        if not self.domino_api_proxy:
            return None
        try:
            response = requests.get(f"{self.domino_api_proxy}/access-token", timeout=10)
            response.raise_for_status()
            return response.text.strip()
        except Exception as e:
            print(f"Warning: Failed to get access token: {e}")
            return None
    
    def _get_headers(self):
        """Build request headers with fresh Bearer token."""
        headers = {"Content-Type": "application/json"}
        token = self._get_token()
        if token:
            headers["Authorization"] = f"Bearer {token}"
        return headers
    
    def _request(self, method: str, endpoint: str, json_data: dict = None, expect_json: bool = True) -> dict | str:
        """Make authenticated request to Qdrant."""
        url = f"{self.url}{endpoint}"
        response = requests.request(
            method=method,
            url=url,
            headers=self._get_headers(),
            json=json_data,
            timeout=30,
        )
        response.raise_for_status()
        if expect_json:
            return response.json()
        return response.text
    
    def health_check(self) -> str:
        """Check Qdrant health (returns plain text)."""
        return self._request("GET", "/healthz", expect_json=False)
    
    def list_collections(self) -> list:
        """List all collections."""
        result = self._request("GET", "/collections")
        return result.get("result", {}).get("collections", [])
    
    def get_collection(self, name: str) -> dict:
        """Get collection info."""
        result = self._request("GET", f"/collections/{name}")
        return result.get("result", {})
    
    def search(
        self,
        collection_name: str,
        query_vector: list,
        limit: int = 5,
        with_payload: bool = True,
        score_threshold: float = None,
        filter_conditions: dict = None,
    ) -> list:
        """
        Perform vector search on collection.
        
        Args:
            collection_name: Name of the collection
            query_vector: Query embedding vector
            limit: Maximum number of results
            with_payload: Include document payload
            score_threshold: Minimum score threshold
            filter_conditions: Qdrant filter object
        
        Returns:
            List of search results
        """
        data = {
            "vector": query_vector,
            "limit": limit,
            "with_payload": with_payload,
        }
        if score_threshold is not None:
            data["score_threshold"] = score_threshold
        if filter_conditions:
            data["filter"] = filter_conditions
        
        result = self._request("POST", f"/collections/{collection_name}/points/search", data)
        return result.get("result", [])
    
    def get_points(self, collection_name: str, point_ids: list) -> list:
        """Retrieve points by IDs."""
        data = {
            "ids": point_ids,
            "with_payload": True,
            "with_vector": False,
        }
        result = self._request("POST", f"/collections/{collection_name}/points", data)
        return result.get("result", [])
    
    def scroll(self, collection_name: str, limit: int = 10, offset=None) -> tuple:
        """
        Scroll through collection points.
        
        Returns:
            Tuple of (points, next_offset)
        """
        data = {
            "limit": limit,
            "with_payload": True,
            "with_vector": False,
        }
        if offset:
            data["offset"] = offset
        
        result = self._request("POST", f"/collections/{collection_name}/points/scroll", data)
        return (
            result.get("result", {}).get("points", []),
            result.get("result", {}).get("next_page_offset"),
        )


# Initialize client
client = QdrantClient(url=QDRANT_URL, domino_api_proxy=DOMINO_API_PROXY)
print("Client initialized (token fetched per request)")

## 3. Health Check

In [ ]:
# Check Qdrant health
health = client.health_check()
print(f"Health check: {health}")

## 4. List Collections

In [ ]:
# List all collections
collections = client.list_collections()

print(f"Found {len(collections)} collections:")
for col in collections:
    print(f"  - {col['name']}")

## 5. Get Collection Info

In [ ]:
# Get details for a specific collection
COLLECTION_NAME = 'ntsb_incidents'  # Change as needed

info = client.get_collection(COLLECTION_NAME)

print(f"Collection: {COLLECTION_NAME}")
print(f"  Status: {info.get('status')}")
print(f"  Points count: {info.get('points_count')}")
print(f"  Vectors count: {info.get('vectors_count')}")

config = info.get('config', {})
params = config.get('params', {})
vectors = params.get('vectors', {})
print(f"  Vector size: {vectors.get('size')}")
print(f"  Distance: {vectors.get('distance')}")

## 6. Generate Embedding for Search

To search, you need a query vector. This example uses local sentence-transformers (BAAI/bge-base-en-v1.5).

In [ ]:
def get_embedding(text, model_name='BAAI/bge-base-en-v1.5'):
    """Generate embedding using local sentence-transformers model."""
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name)
        return model.encode(text).tolist()
    except ImportError:
        print("sentence-transformers not installed. Run: pip install sentence-transformers")
        return None


# Generate embedding for search query
QUERY_TEXT = "engine failure during takeoff"
query_vector = get_embedding(QUERY_TEXT)

if query_vector:
    print(f"Generated embedding for: '{QUERY_TEXT}'")
    print(f"Vector dimensions: {len(query_vector)}")

## 7. Vector Search

In [ ]:
# Perform search
if query_vector:
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=5,
    )
    
    print(f"Search results for: '{QUERY_TEXT}'")
    print(f"Found {len(results)} results\n")
    print("=" * 60)
    
    for i, hit in enumerate(results, 1):
        print(f"\n{i}. Score: {hit['score']:.4f}")
        print(f"   ID: {hit['id']}")
        
        payload = hit.get('payload', {})
        text = payload.get('text', '')[:300]
        print(f"   Text: {text}...")

## 8. Search with Filters

In [ ]:
# Search with filter on injury_severity
if query_vector:
    filter_conditions = {
        'must': [
            {
                'key': 'injury_severity',
                'match': {'value': 'Fatal'}
            }
        ]
    }
    
    filtered_results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=5,
        filter_conditions=filter_conditions,
    )
    
    print(f"Filtered search (Fatal injuries only):")
    print(f"Found {len(filtered_results)} results\n")
    
    for i, hit in enumerate(filtered_results, 1):
        print(f"{i}. Score: {hit['score']:.4f} - ID: {hit['id']}")

## 9. Get Points by ID

In [ ]:
# Get specific points by ID (using IDs from previous search)
if results:
    sample_ids = [hit['id'] for hit in results[:2]]
    
    points = client.get_points(COLLECTION_NAME, sample_ids)
    
    print(f"Retrieved {len(points)} points by ID:")
    for point in points:
        print(f"\nID: {point['id']}")
        payload = point.get('payload', {})
        for key, value in list(payload.items())[:5]:
            if key != 'text':
                print(f"  {key}: {value}")

## 10. Scroll Through Collection

In [ ]:
# Scroll first 5 points
points, next_offset = client.scroll(COLLECTION_NAME, limit=5)

print(f"First 5 points in {COLLECTION_NAME}:")
for point in points:
    print(f"  - {point['id']}")

print(f"\nNext offset: {next_offset}")

## Summary

This notebook demonstrated:

1. **QdrantClient class** - Handles Bearer token auth for Domino reverse proxy
2. **Health checks** - Verify Qdrant connectivity
3. **Collection management** - List and inspect collections
4. **Embedding generation** - Local sentence-transformers (BAAI/bge-base-en-v1.5)
5. **Vector search** - Basic and filtered searches
6. **Point retrieval** - Get specific points by ID
7. **Scrolling** - Paginate through collection

### Qdrant REST API Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/healthz` | GET | Health check |
| `/collections` | GET | List collections |
| `/collections/{name}` | GET | Collection info |
| `/collections/{name}/points/search` | POST | Vector search |
| `/collections/{name}/points` | POST | Get points by ID |
| `/collections/{name}/points/scroll` | POST | Scroll/paginate |